[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kulinskij-tech/Synergetics/blob/main/syn_py_ua/syn_ua_09_chaos_fractals.ipynb)


# 09. Детермінований хаос, відображення і фрактальна структура

_Магістерський курс «Вступ до синергетики» · українська версія_

> **Провідне питання:** Як детерміновані рівняння породжують обмежений рух, що залишається нестійким щодо прогнозування?

**Як працювати з ноутбуком.** Виконуйте комірки послідовно, згори донизу. Якщо не зазначено інше, усі параметри безрозмірні. Пояснюйте зміст кожної діагностики: сама лише ефектна траєкторія ще не доводить хаосу чи самоорганізації.


## Результати навчання

Після опрацювання модуля ви зможете:

- пояснити фізичний зміст змінних і параметрів Лоренца як результату скорочення задачі теплової конвекції;
- відрізняти чутливість до початкових умов від числової похибки та випадковості;
- оцінювати найбільший показник Ляпунова за парою близьких траєкторій;
- пов'язувати відображення повернення, інваріантні міри та фрактальну геометрію.


## Необхідний вступ: потоки, атрактори і ляпуновське зростання

Автономне звичайне диференціальне рівняння породжує неперервний у часі **потік** у просторі станів, а ітераційне правило — дискретне **відображення**. Після завершення перехідного процесу траєкторії можуть наближатися до атрактора: стаціонарної точки, періодичної орбіти, квазіперіодичної множини або хаотичної інваріантної множини. Басейн притягання — це сукупність початкових станів, що прямують до цього атрактора.

Хаос означає детерміновану аперіодичну динаміку з чутливою залежністю, а не просто складний вигляд графіка. Показник Ляпунова вимірює довгочасову експоненційну швидкість зростання нескінченно малого збурення. Додатне зростання потрібно оцінювати з регулярним перенормуванням: дві траєкторії, які вільно розбігаються, зрештою насичуються розміром атрактора і вже не показують локального показника. Змінюйте допуски інтегратора та крок, щоб відділити динаміку від числової похибки.


## Потік Лоренца: теплова конвекція, скорочена до трьох мод

Лоренц отримав свою систему жорстким галеркінським усіченням рівнянь Обербека—Буссінеска для конвекції Релея—Бенара: горизонтального шару рідини, підігрітого знизу. У безрозмірній моделі $x$ пропорційна амплітуді й орієнтації швидкості конвективного вала, $y$ вимірює пов'язаний з ним горизонтальний контраст температури, а $z$ — викривлення вертикального температурного профілю відносно теплопровідного стану. Тому дві «лопаті» атрактора відповідають епізодам із протилежними напрямами циркуляції, а не двом географічним погодним режимам.

$$\dot x=\sigma(y-x),\quad \dot y=x(\rho-z)-y,\quad \dot z=xy-\beta z.$$

Тут $\sigma$ — число Прандтля, $\rho=Ra/Ra_c$ — зведений керувальний параметр Релея, а $\beta$ визначається збереженим просторовим хвильовим числом і геометрією комірки. За $\rho<1$ теплопровідна рівновага стійка. У точці $\rho=1$ вона втрачає стійкість, а за $\rho>1$ з'являються дві симетрично пов'язані конвективні рівноваги:

$$(x_*,y_*,z_*)=\left(\pm\sqrt{\beta(\rho-1)},\,\pm\sqrt{\beta(\rho-1)},\,\rho-1\right).$$

За класичних параметрів $(\sigma,\rho,\beta)=(10,28,8/3)$ стаціонарні вали нестійкі, і траєкторія нерегулярно перемикається між їх околами. Дивергенція дорівнює $-(\sigma+1+\beta)<0$, тому фазові об'єми стискаються. Водночас додатний найбільший показник Ляпунова $\lambda_1$ зумовлює локальне зростання збурення $\|\delta x(t)\|\sim e^{\lambda_1t}\|\delta x(0)\|$. Обмеженість разом із розтягуванням і складанням створює дивний атрактор, а не необмежене розбігання. Це концептуальна лабораторія детермінованої передбачуваності, а не кількісна модель погоди.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.2, 4.2),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
rng = np.random.default_rng(20260904)

from scipy.integrate import solve_ivp

sigma, rho, beta = 10.0, 28.0, 8/3
def lorenz(t, state):
    x, y, z = state
    return sigma*(y-x), x*(rho-z)-y, x*y-beta*z

t = np.linspace(0, 60, 12000)
y0 = np.array([1.0, 1.0, 1.0])
eps0 = 1e-8
s1 = solve_ivp(lorenz, (0, 60), y0, t_eval=t, rtol=1e-10, atol=1e-12)
s2 = solve_ivp(lorenz, (0, 60), y0 + [eps0, 0, 0], t_eval=t, rtol=1e-10, atol=1e-12)
distance = np.linalg.norm(s2.y-s1.y, axis=0)

convective = np.sqrt(beta*(rho-1))
print("conductive equilibrium:", (0.0, 0.0, 0.0))
print("convective equilibria:", (convective, convective, rho-1),
      (-convective, -convective, rho-1))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
mask = t > 10
ax[0].plot(s1.y[0, mask], s1.y[2, mask], lw=0.35)
ax[0].set(xlabel="x", ylabel="z", title="Lorenz attractor projection")
ax[1].semilogy(t, distance)
ax[1].axhline(1, color="k", ls="--", lw=1)
ax[1].set(xlabel="time", ylabel="trajectory separation", title="Growth and saturation")
plt.tight_layout()


In [ ]:
# Logistic-map bifurcation diagram and Lyapunov exponent.
r_values = np.linspace(2.8, 4.0, 1400)
x = 0.37*np.ones_like(r_values)
for _ in range(1200):
    x = r_values*x*(1-x)

rs, xs, lambdas = [], [], np.zeros_like(r_values)
for _ in range(260):
    x = r_values*x*(1-x)
    rs.append(r_values.copy())
    xs.append(x.copy())
    lambdas += np.log(np.maximum(np.abs(r_values*(1-2*x)), 1e-14))
lambdas /= 260

fig, ax = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
ax[0].plot(np.concatenate(rs), np.concatenate(xs), ",k", alpha=0.18)
ax[0].set(ylabel="asymptotic x", title="Logistic-map bifurcation diagram")
ax[1].plot(r_values, lambdas, lw=0.8)
ax[1].axhline(0, color="k", lw=1)
ax[1].set(xlabel="control r", ylabel=r"Lyapunov exponent $\lambda$")
plt.tight_layout()


## Оцінюване завдання

Спочатку перевірте теплопровідну та конвективні рівноваги й дослідіть їх стійкість по обидва боки $\rho=1$, пояснюючи результат як початок конвекції. Потім реалізуйте стандартний алгоритм із перенормуванням для найбільшого показника Ляпунова системи Лоренца за $\rho=28$. Покажіть інтервал експоненційного зростання та перевірте стійкість оцінки до інтервалу перенормування, допуску інтегратора й тривалості відкинутого перехідного процесу.

Подання має містити чітке твердження, його математичне або фізичне обґрунтування, числові свідчення та принаймні одне обмеження висновку.


## Література та атрибуція

**Основне читання з локальної бібліотеки:** Mikhailov and Loskutov, *Foundations of Synergetics II: Complex Patterns*; Strogatz, *Nonlinear Dynamics and Chaos*.

**Перевірені вебджерела:**
- [Оригінальна стаття Лоренца](https://doi.org/10.1175/1520-0469(1963)020%3C0130:DNF%3E2.0.CO;2)
- [Scholarpedia: butterfly effect](https://www.scholarpedia.org/article/Butterfly_effect)
- [Scholarpedia: chaos topology](https://www.scholarpedia.org/article/Chaos_topology)

Локальні книги є наданими матеріалами курсу. Вебпосилання мають додатковий характер; їх перевірено 04.09.2026.
